## Overview
DOM elements react to a variety of events like clicking, keyboard event, scroll events, etc. React allows us to respond to those events:

In [ ]:
function AlertButton({ message }) {
    function handleClick() {
        alert(message);
    }
    
    // Note that onClick prop is different from onclick DOM attribute
    return (
        <button onClick={handleClick}>
            Click Me
        </button>
    );
}

The event generated is instance of `SyntheticEvent`. What does this mean? React does not attach an event listener to the button like:

In [ ]:
button.addEventListener('click', handleClick);

Instead it creates one single event listener attached to the root element. When a DOM element like a button is clicked it generates native event bubbling throught the DOM tree till it reaches the root element and triggering the event handler. React looks at the source of the event and identifies which component it came from and wraps the native event inside `SyntheticEvent`. Then it triggers the handler associated with that event and component.

`SyntheticEvent` ensures a common browser-agnostic interface to events. It has the same structure as native event which means we still have access to methods like `preventDefault()` and `stopPropagation()`.

React supports the following events:

| Event Type/Category | Events                                                                           |
|---------------------|----------------------------------------------------------------------------------|
| Clipboard           | onCopy, onCut, onPaste                                                           |
| Composition         | onCompositionEnd, onCompositionStart, onCompositionUpdate                        |
| Keyboard            | onKeyDown, onKeyPress, onKeyUp                                                   | 
| Focus               | onChange, onInput, onSubmit                                                      |
| Form                | onFocus, onBlur                                                                  |
| Mouse               | onClick, onContextMenu, onDoubleClick, onDrag, onDragEnd, onDragEnter, onDragExit, onDragLeave, onDragOver, onDragStart, onDrop, onMouseDown, onMouseEnter, onMouseLeave, onMouseMove, onMouseOut, onMouseOver, onMouseUp                 |
| Selection           | onSelect                                                                         |
| Touch               | onTouchCancel, onTouchEnd, onTouchMove, onTouchStart                             |
| UI                  | onScroll                                                                         |
| Wheel               | onWheel                                                                          |
| Media               | onAbort, onCanPlay, onCanPlayThrough, onDurationChange, onEmptied, onEncrypted, onEnded, onError, onLoadedData, onLoadedMetadata, onLoadStart, onPause, onPlay, onPlaying, onProgress, onRateChange, onSeeked, onSeeking, onStalled, onSuspend, onTimeUpdate, onVolumeChange, onWaiting                                                                                |
| Image               | onLoad, onError                                                                  |
| Animation           | onAnimationStart, onAnimationEnd, onAnimationIteration                           |
| Transition          | onTransitionEnd                                                                  |

Events in React are triggered on the bubbling phase (inside to outside). To trigger an event on the capturing phase (outside to inside) add the word "Capture" to the event name (e.g., `onClick` would become `onClickCapture`).

One difference that React event listener functions have, is that we cannot return `false` to prevent default behavior. We must call `preventDefault` explicitly.

## Component Events
React components can also declare their own custom events. For example, the `List` component below exposes an `onRemove` event:

In [ ]:
const List = ({ items, onRemove }) => {
  return (
    <ul>
      {items.map((item) => (
        <li key={item.id}>
          {item.text} 
          <button onClick={() => onRemove(item.id)}>
            Remove
          </button>
        </li>
      ))}
    </ul>
  );
};

Now the parent component passes a handler to the `onRemove` event:

In [ ]:
const [data, setData] = useState([
    { id: 1, text: 'Learn React Fragments' },
    { id: 2, text: 'Master Synthetic Events' },
    { id: 3, text: 'Understand Context API' },
  ]);

const handleRemove = (id) => {
    const updatedData = data.filter(item => item.id !== id);
    setData(updatedData);
    console.log(`Removed item with ID: ${id}`);
};

<List items={data} onRemove={handleRemove} />

Event handlers don't need to be pure. These are good place to change something or make API calls, etc:

In [ ]:
const [loading, setLoading] = useState(false);
const [status, setStatus] = useState('');

const handleSave = async (data) => {
    setLoading(true);
    setStatus('Saving');

    try{
        const response = await fetch('https://example.api.com/save', {
            method: 'POST',
            body: JSON.stringify({d: data})
        });
    
        if(response.ok) {
            setStatus('Success');    
        } else {
            setStatus('Failure');
        }
    } catch {
        setStatus('Failure');
    } finally {
        setLoading(false);
    }
}

<Profile onSave={handleSave} />

In older versions of React (v16 and below), React reused event objects to save memory. This meant if we tried to access `e.target` after an `await`, it would be `null` because React had already cleared the object.
- Modern React (v17+): This is no longer an issue. We can access the event object safely after an `await`.
- Older React: We would have to call `e.persist()` at the start of the function.